In [0]:
%sql
CREATE TABLE IF NOT EXISTS audit_validation
(
    run_id STRING,
    table_name STRING,
    source_count BIGINT,
    target_count BIGINT,
    rejected_count BIGINT,
    duplicate_count BIGINT,
    load_status STRING,
    load_time TIMESTAMP
)
USING DELTA;

In [0]:
%sql
SELECT *
FROM audit_validation;

In [0]:
source_count = spark.table("bronze_customers").count()
target_count = spark.table("silver_customers").count()

print("Source :", source_count)
print("Target :", target_count)

In [0]:
from datetime import datetime

status = "SUCCESS" if source_count == target_count else "FAILED"

spark.sql(f"""
INSERT INTO audit_validation (table_name, validation_name, source_count, target_count, status, remarks, validation_time)
VALUES
(
    'silver_customers',
    'RUN_001',
    {source_count},
    {target_count},
    '{status}',
    'Record count validation',
    current_timestamp()
)
""")

In [0]:
%sql
SELECT *
FROM audit_validation
ORDER BY validation_time DESC;

In [0]:
fake_target_count = target_count - 2

status = "SUCCESS" if source_count == fake_target_count else "FAILED"

print(status)

In [0]:
%sql
SELECT
    table_name,
    source_count,
    target_count,
    status,
    validation_time
FROM audit_validation
ORDER BY validation_time DESC;